<a href="https://colab.research.google.com/github/bhanukiran29/BISLAB_1WN24CS150/blob/main/BIS_LAB6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import math

# ============================================
# CUCKOO SEARCH ALGORITHM - KNAPSACK PROBLEM
# ============================================

np.random.seed(42)

# Item weights and values
weights = np.array([10, 20, 30, 15, 25, 12, 18, 8, 22, 14])
values  = np.array([60, 100, 120, 75, 90, 65, 85, 40, 95, 70])

CAPACITY = 50
NESTS = 25
ITERATIONS = 100
PA = 0.25
BETA = 1.5
ALPHA = 0.01

N = len(weights)

# ============================================
# FITNESS FUNCTION
# ============================================

def repair(solution):
    """Remove items until the capacity constraint is satisfied."""

    solution = solution.copy()

    while np.sum(solution * weights) > CAPACITY:
        selected = np.where(solution == 1)[0]

        # Remove item with lowest value-to-weight ratio
        worst = min(
            selected,
            key=lambda i: values[i] / weights[i]
        )

        solution[worst] = 0

    return solution


def fitness(solution):
    solution = repair(solution)
    return np.sum(solution * values)


def total_weight(solution):
    return np.sum(solution * weights)


# ============================================
# LEVY FLIGHT
# ============================================

def levy_flight():
    sigma = (
        math.gamma(1 + BETA) *
        np.sin(np.pi * BETA / 2) /
        (
            math.gamma((1 + BETA) / 2) *
            BETA *
            2 ** ((BETA - 1) / 2)
        )
    ) ** (1 / BETA)

    u = np.random.normal(0, sigma, N)
    v = np.random.normal(0, 1, N)

    return u / (np.abs(v) ** (1 / BETA) + 1e-10)


# ============================================
# BINARY CONVERSION
# ============================================

def convert_to_binary(position):
    probability = 1 / (1 + np.exp(-np.clip(position, -10, 10)))

    solution = (np.random.random(N) < probability).astype(int)

    return repair(solution)


# ============================================
# INITIALIZE HOST NESTS
# ============================================

nests = np.random.uniform(-2, 2, (NESTS, N))

best_solution = None
best_value = -1

# Store convergence values
convergence = []

# Initial population evaluation
for nest in nests:
    solution = convert_to_binary(nest)
    score = fitness(solution)

    if score > best_value:
        best_value = score
        best_solution = solution.copy()


# ============================================
# MAIN CUCKOO SEARCH ALGORITHM
# ============================================

print("=" * 68)
print("          CUCKOO SEARCH - 0/1 KNAPSACK PROBLEM")
print("=" * 68)

print(f"Items: {N} | Nests: {NESTS} | Iterations: {ITERATIONS}")
print(f"Bag Capacity: {CAPACITY} kg | Discovery Probability: {PA}")
print("-" * 68)

print(
    f"{'ITERATION':<15}"
    f"{'CURRENT BEST':<18}"
    f"{'GLOBAL BEST':<18}"
    f"{'STATUS'}"
)

print("-" * 68)

for iteration in range(1, ITERATIONS + 1):

    scores = np.array([
        fitness(convert_to_binary(nest))
        for nest in nests
    ])

    # Generate new cuckoo solutions using Levy Flight
    for i in range(NESTS):

        step = levy_flight()

        new_position = nests[i] + ALPHA * step * (
            nests[i] - best_solution
        )

        new_position = np.clip(new_position, -6, 6)

        new_solution = convert_to_binary(new_position)
        new_score = fitness(new_solution)

        # Select a random host nest
        j = np.random.randint(NESTS)

        if new_score > scores[j]:
            nests[j] = new_position
            scores[j] = new_score

        # Update global best
        if new_score > best_value:
            best_value = new_score
            best_solution = new_solution.copy()

    # Abandon a fraction of worst nests
    abandon_count = int(PA * NESTS)

    scores = np.array([
        fitness(convert_to_binary(nest))
        for nest in nests
    ])

    worst_indices = np.argsort(scores)[:abandon_count]

    for index in worst_indices:
        nests[index] = np.random.uniform(-2, 2, N)

    # Preserve the best solution (elitism)
    best_index = np.argmin(scores)
    nests[best_index] = best_solution.copy().astype(float)

    # Record convergence
    convergence.append(best_value)

    # Display every 10 iterations
    if iteration % 10 == 0:

        current_best = max(
            fitness(convert_to_binary(nest))
            for nest in nests
        )



        status = (
    "Improved"
    if current_best >= best_value
    else "-"
)

        print(
            f"{iteration:<15}"
            f"{current_best:<18}"
            f"{best_value:<18}"
            f"{status}"
        )


# ============================================
# FINAL RESULTS
# ============================================

best_solution = repair(best_solution)

selected_items = np.where(best_solution == 1)[0]

final_weight = total_weight(best_solution)
final_value = fitness(best_solution)

print("\n" + "=" * 68)
print("                  OPTIMIZATION RESULTS")
print("=" * 68)

print(f"{'ITEM':<12}{'WEIGHT (KG)':<18}{'VALUE':<15}")
print("-" * 68)

for i in selected_items:
    print(
        f"Item {i + 1:<7}"
        f"{weights[i]:<18}"
        f"{values[i]:<15}"
    )

print("-" * 68)

print(f"Selected Items      : {len(selected_items)}")
print(f"Total Weight        : {final_weight} kg")
print(f"Maximum Capacity    : {CAPACITY} kg")
print(f"Total Value         : {final_value}")
print(f"Unused Capacity     : {CAPACITY - final_weight} kg")

print("=" * 68)



          CUCKOO SEARCH - 0/1 KNAPSACK PROBLEM
Items: 10 | Nests: 25 | Iterations: 100
Bag Capacity: 50 kg | Discovery Probability: 0.25
--------------------------------------------------------------------
ITERATION      CURRENT BEST      GLOBAL BEST       STATUS
--------------------------------------------------------------------
10             250               265               -
20             240               265               -
30             265               265               Improved
40             265               265               Improved
50             265               265               Improved
60             255               265               -
70             265               265               Improved
80             265               265               Improved
90             265               265               Improved
100            250               265               -

                  OPTIMIZATION RESULTS
ITEM        WEIGHT (KG)       VALUE          
---------